<a href="https://colab.research.google.com/github/jonasknoll57/Bachelorarbeit_Demand-AD/blob/main/V5_AD_Only_Mannheim_Daily.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Zugangsberechtigung auf Drive

from google.colab import drive
drive.mount('/content/drive')

# Gezippte Daten werden entpackt und in hohes Verzeichnis gelegt

!cp "/content/drive/MyDrive/BA_Colab/data.zip" "/content/data.zip"

!unzip -q "/content/data.zip" -d "/content"

!rm "/content/data.zip"
!rm "/content/_MACOSX"


Mounted at /content/drive
rm: cannot remove '/content/_MACOSX': No such file or directory


In [4]:
# ══════════════════════════════════════════════════════════════
# 0 — Setup & Imports
# ══════════════════════════════════════════════════════════════
import sympy.printing

import os, glob, re, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from shapely import wkb
import sympy.printing

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_recall_curve, auc, f1_score, roc_auc_score,
    classification_report, roc_curve
)
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cpu


In [6]:
# ══════════════════════════════════════════════════════════════
# CELL 1 — Konfiguration (GEÄNDERT)
# ══════════════════════════════════════════════════════════════

# ── Pfade ──
DEMAND_BASE        = '/content/data/demand'
STATION_NAMES_PATH = '/content/data/station_names/station_names.parquet'
GEO_INFO_PATH      = '/content/data/geo_information/geo_information.parquet'
WEATHER_PATH       = '/content/data/weather/weather.parquet'
HOLIDAYS_PATH      = '/content/data/holidays/holidays.parquet'
VACATIONS_PATH     = '/content/data/vacations/vacations.parquet'

OUTPUT_PATH        = '/content/data/V5_AD_Mannheim_daily.parquet'

# ── Konfiguration ──
CITY               = 'Mannheim'
NETWORK_NAME       = 'Mannheim'
WEATHER_STATION_ID = 292348
FEDERAL_STATE      = 'Baden-Württemberg'

# ── AD-Hyperparameter ──
WINDOW_SIZE        = 7                    # TAGE (1 Woche) statt 24 Stunden
LATENT_DIM         = 32

# ── Labeling-Schwellen ──
Z_TRAIN_THRESHOLD   = 2.0
Z_ANOMALY_THRESHOLD = 3.0
MIN_EVENTS_PER_DAY  = 3.0                # Stationsfilter: mind. 3 Events/Tag
MIN_HIST_MEAN       = 5.0                # Tages-Anomalie: hist_mean >= 5 (höher als hourly!)
MIN_ABSOLUTE         = 10                 # Tages-Anomalie: mind. 10 Events absolut

# ── Active-Hours Filtering → wird zu Active-Days Filtering ──
FILTER_ZERO_DAYS   = True                # Tage mit demand=0 aus Training entfernen

# ── Point-wise Scoring ──
SCORE_LAST_N_STEPS = 1                   # Letzter Tag der Sequenz für Score (bei 7-Tage-Window)

# ── Training ──
BATCH_SIZE         = 4096
EPOCHS             = 50
LEARNING_RATE      = 1e-3
TRAIN_RATIO        = 0.67
VAL_RATIO          = 0.83

# ══════════════════════════════════════════════════════════════
# [DESIGN-ENTSCHEIDUNG] Daily Aggregation
#
# BEGRÜNDUNG:
#   Stündliche Daten pro Station sind zu granular für AE-basierte AD:
#   - ~55% Zero-Demand → AE lernt primär "rekonstruiere Null"
#   - Anomalien (Stunde: 5 statt 2 Events) sind im Rauschen kaum trennbar
#   - Signal-Rausch-Verhältnis zu niedrig für brauchbare Precision
#
#   Tagesaggregation löst diese Probleme:
#   - Zero-Inflation verschwindet (aktive Stationen haben täglich >0)
#   - Anomalien werden klarer (Tag: 50 statt 20 Trips = deutliches Signal)
#   - Praxisrelevant: Netzwerkmanager interessieren tägliche Muster
#   - Perfekt transferierbar: Tagesmuster existieren in jeder Stadt
#
#   Window = 7 Tage (1 Woche) → erfasst wöchentliche Periodizität
# ══════════════════════════════════════════════════════════════

FEATURE_COLS = [
    # ── Tägliche Nachfrage (roh) ──
    'total_demand', 'n_lends', 'n_returns',
    # ── Temporale Lags (roh, auf Tagesebene) ──
    'demand_lag_1d',                      # Gestern
    'demand_lag_7d',                      # Gleicher Wochentag letzte Woche
    'demand_lag_14d',                     # Gleicher Wochentag vor 2 Wochen
    # ── Rolling Features (Kontext ohne Leakage) ──
    'demand_rolling_7d_mean',             # Durchschnitt der letzten 7 Tage
    'demand_rolling_7d_std',              # Volatilität der letzten 7 Tage
    # ── Kalender (zyklisch) ──
    'dow_sin', 'dow_cos',                 # Wochentag
    'month_sin', 'month_cos',             # Monat
    # ── Kalender (binär) ──
    'is_weekend', 'is_holiday', 'is_vacation',
    # ── Wetter (Tagesdurchschnitt/-summe) ──
    'temperature', 'humidity', 'precipitation', 'wind_speed',
]

DEMAND_FEATURE_INDICES = list(range(8))   # Indices 0-7: demand + lags + rolling

N_FEATURES     = len(FEATURE_COLS)
INPUT_DIM_FLAT = WINDOW_SIZE * N_FEATURES

print(f'═══ V5 Konfiguration (Daily) ═══')
print(f'Aggregation:  DAILY (statt hourly)')
print(f'Window:       {WINDOW_SIZE} Tage (1 Woche)')
print(f'Features:     {N_FEATURES} → {FEATURE_COLS}')
print(f'Demand-Features für Scoring: {[FEATURE_COLS[i] for i in DEMAND_FEATURE_INDICES]}')
print(f'Input dim flat: {INPUT_DIM_FLAT}')

═══ V5 Konfiguration (Daily) ═══
Aggregation:  DAILY (statt hourly)
Window:       7 Tage (1 Woche)
Features:     19 → ['total_demand', 'n_lends', 'n_returns', 'demand_lag_1d', 'demand_lag_7d', 'demand_lag_14d', 'demand_rolling_7d_mean', 'demand_rolling_7d_std', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend', 'is_holiday', 'is_vacation', 'temperature', 'humidity', 'precipitation', 'wind_speed']
Demand-Features für Scoring: ['total_demand', 'n_lends', 'n_returns', 'demand_lag_1d', 'demand_lag_7d', 'demand_lag_14d', 'demand_rolling_7d_mean', 'demand_rolling_7d_std']
Input dim flat: 133


In [7]:
# ══════════════════════════════════════════════════════════════
# 2 — Hilfsdaten laden (Stationen, Wetter, Feiertage)
# ══════════════════════════════════════════════════════════════

# ── Station Names & Typ-Klassifikation ──
def classify_station(name: str) -> str:
    if not isinstance(name, str) or name.strip() == '':
        return 'unknown'
    n = name.strip()
    if re.search(r'(?i)^recording[_\-\s]', n):  return 'recording'
    if re.match(r'(?i)^bike[-_]?\s*\d*', n):    return 'bike'
    if re.search(r'(?i)(virtuell|virtual)', n):  return 'virtual'
    if re.fullmatch(r'[\d\s\-_\.#/]+', n):      return 'only_nums'
    return 'real'

station_names = pd.read_parquet(STATION_NAMES_PATH)
station_names = station_names.rename(columns={'id': 'station_name_id', 'name': 'station_name'})
station_names['station_type'] = station_names['station_name'].apply(classify_station)
type_lookup = station_names.set_index('station_name_id')['station_type'].to_dict()
print(f'Station-Types: {station_names["station_type"].value_counts().to_dict()}')

# ── Wetter laden & auf Stunde aggregieren ──
weather = pd.read_parquet(WEATHER_PATH)
weather['timestamp'] = pd.to_datetime(weather['timestamp'], utc=True)
weather_ma = weather[weather['location_id'] == WEATHER_STATION_ID].copy()
weather_ma['hour_ts'] = weather_ma['timestamp'].dt.floor('h')

weather_hourly = (
    weather_ma.groupby('hour_ts')
    .agg(
        temperature   = ('temperature', 'mean'),
        humidity      = ('humidity', 'mean'),
        precipitation = ('precipitation', 'sum'),
        wind_speed    = ('wind_speed', 'mean'),
    )
    .reset_index()
)
print(f'Wetter stündlich: {len(weather_hourly):,} Zeilen | '
      f'{weather_hourly["hour_ts"].min().date()} – {weather_hourly["hour_ts"].max().date()}')

# ── Feiertage & Ferien (nur BaWü) ──
holidays  = pd.read_parquet(HOLIDAYS_PATH)
vacations = pd.read_parquet(VACATIONS_PATH)
holidays['start_date']  = pd.to_datetime(holidays['start_date'])
holidays['end_date']    = pd.to_datetime(holidays['end_date'])
vacations['start_date'] = pd.to_datetime(vacations['start_date'])
vacations['end_date']   = pd.to_datetime(vacations['end_date'])

holidays_bw  = holidays[holidays['federal_state_name'] == FEDERAL_STATE]
vacations_bw = vacations[vacations['federal_state_name'] == FEDERAL_STATE]

holiday_dates = set()
for _, row in holidays_bw.iterrows():
    for d in pd.date_range(row['start_date'], row['end_date']):
        holiday_dates.add(d.date())

vacation_dates = set()
for _, row in vacations_bw.iterrows():
    for d in pd.date_range(row['start_date'], row['end_date']):
        vacation_dates.add(d.date())

print(f'Feiertage BaWü: {len(holiday_dates)} Tage | Ferien BaWü: {len(vacation_dates)} Tage')


# ══════════════════════════════════════════════════════════════
# CELL 3 — Demand laden & auf TAG aggregieren (GEÄNDERT)
# ══════════════════════════════════════════════════════════════

files = glob.glob(os.path.join(DEMAND_BASE, CITY, '**', '*.parquet'), recursive=True)
if not files:
    files = glob.glob(os.path.join(DEMAND_BASE, CITY, '*.parquet'))
print(f'Parquet-Dateien gefunden: {len(files)}')

cols = ['network_name', 'timestamp', 'station_id', 'station_name_id',
        'location_id', 'n_lends', 'n_returns']
demand_raw = pd.concat([pd.read_parquet(f, columns=cols) for f in files], ignore_index=True)
demand_raw['timestamp'] = pd.to_datetime(demand_raw['timestamp'], utc=True)

demand_raw['station_type'] = demand_raw['station_name_id'].map(type_lookup).fillna('unknown')
demand = demand_raw[demand_raw['station_type'] == 'real'].copy()

print(f'Demand roh:      {len(demand_raw):,} Zeilen')
print(f'Demand (real):   {len(demand):,} Zeilen')
print(f'Stationen:       {demand["station_id"].nunique()}')
print(f'Zeitraum:        {demand["timestamp"].min().date()} – {demand["timestamp"].max().date()}')

# ── TÄGLICHE Aggregation pro Station ──
demand['date'] = demand['timestamp'].dt.date
demand['date'] = pd.to_datetime(demand['date'], utc=True)

daily = (
    demand
    .groupby(['station_id', 'date'])
    .agg(
        n_lends         = ('n_lends', 'sum'),
        n_returns       = ('n_returns', 'sum'),
        station_name_id = ('station_name_id', 'first'),
        location_id     = ('location_id', 'first'),
    )
    .reset_index()
)
daily['total_demand'] = daily['n_lends'] + daily['n_returns']
print(f'Täglich aggregiert: {len(daily):,} Zeilen')

# ── Lücken füllen: Jede Station braucht JEDEN Tag ──
all_days = pd.date_range(
    daily['date'].min(), daily['date'].max(), freq='D', tz='UTC'
)

station_info = (
    daily.groupby('station_id')
    .agg(station_name_id=('station_name_id', 'first'), location_id=('location_id', 'first'))
    .reset_index()
)

full_index = pd.MultiIndex.from_product(
    [station_info['station_id'].values, all_days],
    names=['station_id', 'date']
)
daily_full = (
    daily[['station_id', 'date', 'n_lends', 'n_returns', 'total_demand']]
    .set_index(['station_id', 'date'])
    .reindex(full_index, fill_value=0)
    .reset_index()
)
daily_full = daily_full.merge(station_info, on='station_id', how='left')

print(f'Nach Lückenfüllung: {len(daily_full):,} Zeilen')
print(f'Tage pro Station:  {len(all_days):,}')
print(f'Stationen:         {daily_full["station_id"].nunique()}')

# ── Stationsfilter ──
n_days = len(all_days)
station_activity = daily_full.groupby('station_id')['total_demand'].sum()
min_events = int(n_days * MIN_EVENTS_PER_DAY)
active_stations = station_activity[station_activity >= min_events].index

daily_full = daily_full[daily_full['station_id'].isin(active_stations)].copy()
print(f'Stationen nach Filter (≥{MIN_EVENTS_PER_DAY}/Tag): '
      f'{daily_full["station_id"].nunique()} '
      f'(entfernt: {len(station_activity) - len(active_stations)})')

# ── Demand-Statistik (Vergleich zu hourly) ──
print(f'\n═══ Daily Demand Statistik ═══')
print(f'Mean:   {daily_full["total_demand"].mean():.1f}')
print(f'Median: {daily_full["total_demand"].median():.0f}')
print(f'Std:    {daily_full["total_demand"].std():.1f}')
print(f'Zero-Tage: {(daily_full["total_demand"]==0).mean():.1%} (vs. ~55% bei hourly)')


# ══════════════════════════════════════════════════════════════
# CELL 4 — Feature Engineering (GEÄNDERT — Daily Features)
# ══════════════════════════════════════════════════════════════

# ── Kalender-Features (kein hour_sin/cos mehr — macht auf Tagesebene keinen Sinn) ──
daily_full['day_of_week'] = daily_full['date'].dt.dayofweek
daily_full['month']       = daily_full['date'].dt.month
daily_full['is_weekend']  = (daily_full['day_of_week'] >= 5).astype(np.int8)
daily_full['is_holiday']  = daily_full['date'].dt.date.apply(
    lambda x: 1 if x in holiday_dates else 0
).astype(np.int8)
daily_full['is_vacation'] = daily_full['date'].dt.date.apply(
    lambda x: 1 if x in vacation_dates else 0
).astype(np.int8)

# Zyklische Kodierung
daily_full['dow_sin']   = np.sin(2 * np.pi * daily_full['day_of_week'] / 7)
daily_full['dow_cos']   = np.cos(2 * np.pi * daily_full['day_of_week'] / 7)
daily_full['month_sin'] = np.sin(2 * np.pi * daily_full['month'] / 12)
daily_full['month_cos'] = np.cos(2 * np.pi * daily_full['month'] / 12)
print(f'Kalender-Features hinzugefügt.')

# ── Wetter-Features (Tagesdurchschnitt) ──
weather_daily = (
    weather_hourly
    .assign(date=weather_hourly['hour_ts'].dt.floor('D'))
    .groupby('date')
    .agg(
        temperature   = ('temperature', 'mean'),
        humidity      = ('humidity', 'mean'),
        precipitation = ('precipitation', 'sum'),     # Tagessumme!
        wind_speed    = ('wind_speed', 'mean'),
    )
    .reset_index()
)
# Timezone anpassen
weather_daily['date'] = pd.to_datetime(weather_daily['date'], utc=True)

daily_full = daily_full.merge(weather_daily, on='date', how='left')
for col in ['temperature', 'humidity', 'precipitation', 'wind_speed']:
    daily_full[col] = (
        daily_full.groupby('station_id')[col]
        .transform(lambda x: x.interpolate(method='linear', limit=3).ffill().bfill())
    )
    median_val = daily_full[col].median()
    daily_full[col] = daily_full[col].fillna(median_val)

print(f'Wetter-Coverage: {daily_full["temperature"].notna().mean()*100:.1f}%')

# ── Lag-Features (TAGESEBENE) ──
daily_full = daily_full.sort_values(['station_id', 'date']).reset_index(drop=True)

for lag_name, lag_days in [('lag_1d', 1), ('lag_7d', 7), ('lag_14d', 14)]:
    daily_full[f'demand_{lag_name}'] = (
        daily_full.groupby('station_id')['total_demand'].shift(lag_days)
    )

# ── Rolling Features (7-Tage Fenster, kein Leakage weil shift(1) zuerst) ──
# WICHTIG: .shift(1) stellt sicher, dass nur vergangene Werte einfließen
daily_full['demand_rolling_7d_mean'] = (
    daily_full.groupby('station_id')['total_demand']
    .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
)
daily_full['demand_rolling_7d_std'] = (
    daily_full.groupby('station_id')['total_demand']
    .transform(lambda x: x.shift(1).rolling(7, min_periods=1).std().fillna(0))
)

before = len(daily_full)
daily_full = daily_full.dropna(subset=['demand_lag_14d']).reset_index(drop=True)
print(f'Lag-Features: {before - len(daily_full):,} Zeilen entfernt (Anlaufphase)')
print(f'Verbleibend: {len(daily_full):,} Zeilen')


# ══════════════════════════════════════════════════════════════
# CELL 5 — Temporaler Split + Labeling (GEÄNDERT — daily)
# ══════════════════════════════════════════════════════════════

t_min = daily_full['date'].min()
t_max = daily_full['date'].max()
total_days = (t_max - t_min).days

train_end = t_min + pd.Timedelta(days=int(total_days * TRAIN_RATIO))
val_end   = t_min + pd.Timedelta(days=int(total_days * VAL_RATIO))

print(f'\n═══ Temporaler Split ═══')
print(f'Train-Ende: {train_end.date()}')
print(f'Val-Ende:   {val_end.date()}')
print(f'Test-Ende:  {t_max.date()}')

# ── Stats NUR aus Trainingsperiode (kein temporales Leakage) ──
train_period = daily_full[daily_full['date'] < train_end]

stats = (
    train_period
    .groupby(['station_id', 'day_of_week'])['total_demand']
    .agg(['mean', 'std'])
    .reset_index()
    .rename(columns={'mean': 'hist_mean', 'std': 'hist_std'})
)

daily_full = daily_full.merge(
    stats, on=['station_id', 'day_of_week'], how='left'
)

# Fallback für Stationen/Slots ohne Trainings-Stats
global_mean = train_period['total_demand'].mean()
global_std  = train_period['total_demand'].std()
daily_full['hist_mean'] = daily_full['hist_mean'].fillna(global_mean)
daily_full['hist_std']  = daily_full['hist_std'].fillna(global_std)
daily_full['hist_std']  = daily_full['hist_std'].clip(lower=0.5)  # Etwas höher bei daily

# z_Score (NUR für Labeling!)
daily_full['z_score'] = (
    (daily_full['total_demand'] - daily_full['hist_mean']) / daily_full['hist_std']
)

print(f'Stats berechnet auf {len(train_period):,} Trainings-Zeilen')

# ── Labeling ──
daily_full['label'] = 'normal'

is_anomaly = (
    (daily_full['z_score'].abs() > Z_ANOMALY_THRESHOLD) &
    (daily_full['hist_mean'] >= MIN_HIST_MEAN) &
    (daily_full['total_demand'] >= MIN_ABSOLUTE)
)
is_grauzone = (
    (daily_full['z_score'].abs() > Z_TRAIN_THRESHOLD) &
    ~is_anomaly
)

daily_full.loc[is_grauzone, 'label'] = 'grauzone'
daily_full.loc[is_anomaly, 'label']  = 'anomal'

print('\n═══ Label-Verteilung (Daily) ═══')
print(daily_full['label'].value_counts())
print(f'Anomalie-Rate: {is_anomaly.mean():.4f}')

anom = daily_full[daily_full['label'] == 'anomal']
if len(anom) > 0:
    print(f'\nAnomalie-Qualität:')
    print(f'  hist_mean:     min={anom["hist_mean"].min():.1f}, '
          f'median={anom["hist_mean"].median():.1f}, max={anom["hist_mean"].max():.1f}')
    print(f'  total_demand:  min={anom["total_demand"].min()}, '
          f'median={anom["total_demand"].median():.0f}, max={anom["total_demand"].max()}')
    print(f'  z_score:       min={anom["z_score"].min():.1f}, '
          f'median={anom["z_score"].median():.1f}, max={anom["z_score"].max():.1f}')

    print(f'\nBeispiele (Top-10 z_score):')
    print(anom.nlargest(10, 'z_score')[
        ['station_id', 'date', 'total_demand', 'hist_mean', 'hist_std', 'z_score']
    ].to_string(index=False))


Station-Types: {'bike': 44747, 'real': 13040, 'virtual': 4827, 'recording': 1972, 'only_nums': 51}
Wetter stündlich: 24,913 Zeilen | 2023-04-01 – 2026-02-02
Feiertage BaWü: 167 Tage | Ferien BaWü: 277 Tage
Parquet-Dateien gefunden: 36
Demand roh:      2,683,584 Zeilen
Demand (real):   2,579,329 Zeilen
Stationen:       128
Zeitraum:        2023-03-31 – 2026-02-02
Täglich aggregiert: 94,410 Zeilen
Nach Lückenfüllung: 133,120 Zeilen
Tage pro Station:  1,040
Stationen:         128
Stationen nach Filter (≥3.0/Tag): 92 (entfernt: 36)

═══ Daily Demand Statistik ═══
Mean:   36.6
Median: 23
Std:    46.0
Zero-Tage: 8.4% (vs. ~55% bei hourly)
Kalender-Features hinzugefügt.
Wetter-Coverage: 100.0%
Lag-Features: 1,288 Zeilen entfernt (Anlaufphase)
Verbleibend: 94,392 Zeilen

═══ Temporaler Split ═══
Train-Ende: 2025-02-28
Val-Ende:   2025-08-11
Test-Ende:  2026-02-02
Stats berechnet auf 63,112 Trainings-Zeilen

═══ Label-Verteilung (Daily) ═══
label
normal      88910
grauzone     4853
anomal      

In [9]:


# ══════════════════════════════════════════════════════════════
# 6 — Labeling (z_Score-basiert, mit Qualitätsfilter)
#
# [DESIGN-ENTSCHEIDUNG] z_Score als Labeling-Methode:
#   - In der AD-Literatur etabliert für unsupervised Anomaly Detection
#   - Wird in der BA als "statistisches Pseudo-Label" deklariert
#   - LIMITATION: Approximation, kein Ground-Truth → in Diskussion adressieren
#
# z_Score wird NUR hier verwendet und NICHT in den Feature-Vektor aufgenommen.
# ══════════════════════════════════════════════════════════════


daily_full['label'] = 'normal'              # ← daily_full


is_anomaly = (
    (daily_full['z_score'].abs() > Z_ANOMALY_THRESHOLD) &
    (daily_full['hist_mean'] >= MIN_HIST_MEAN) &
    (daily_full['total_demand'] >= MIN_ABSOLUTE)
)
is_grauzone = (
    (daily_full['z_score'].abs() > Z_TRAIN_THRESHOLD) &
    ~is_anomaly
)


daily_full.loc[is_grauzone, 'label'] = 'grauzone'   # ← daily_full
daily_full.loc[is_anomaly, 'label']  = 'anomal'     # ← daily_full


print('\n═══ Label-Verteilung ═══')
print(daily_full['label'].value_counts())           # ← daily_full
print(f'Anomalie-Rate: {is_anomaly.mean():.4f}')


# Qualitätscheck
anom = daily_full[daily_full['label'] == 'anomal']  # ← daily_full
print(f'\nAnomalie-Qualität:')
print(f'  hist_mean:     min={anom["hist_mean"].min():.1f}, '
      f'median={anom["hist_mean"].median():.1f}, max={anom["hist_mean"].max():.1f}')
print(f'  total_demand:  min={anom["total_demand"].min()}, '
      f'median={anom["total_demand"].median():.0f}, max={anom["total_demand"].max()}')
print(f'  z_score:       min={anom["z_score"].min():.1f}, '
      f'median={anom["z_score"].median():.1f}, max={anom["z_score"].max():.1f}')



═══ Label-Verteilung ═══
label
normal      88910
grauzone     4853
anomal        629
Name: count, dtype: int64
Anomalie-Rate: 0.0067

Anomalie-Qualität:
  hist_mean:     min=5.0, median=21.2, max=249.2
  total_demand:  min=15, median=69, max=769
  z_score:       min=3.0, median=3.4, max=11.7


In [10]:
# ══════════════════════════════════════════════════════════════
# Speichern (mit Metadaten für Multi-City)
# ══════════════════════════════════════════════════════════════

'''
daily_full['network_name'] = NETWORK_NAME      # ← daily_full
daily_full.to_parquet(OUTPUT_PATH, index=False)  # ← daily_full
print(f'\n✅ Gespeichert: {OUTPUT_PATH}')
print(f'   Shape: {daily_full.shape}')            # ← daily_full
'''


# ══════════════════════════════════════════════════════════════
# CELL 7 — Split & Sequenzen (GEÄNDERT — daily_full, date statt hour_ts)
# ══════════════════════════════════════════════════════════════

# Train: NUR normale Tage
df_train = daily_full[
    (daily_full['date'] < train_end) &
    (daily_full['label'] == 'normal')
].copy()

# [V5] Active-Days Filtering
if FILTER_ZERO_DAYS:
    before_zero = len(df_train)
    df_train = df_train[df_train['total_demand'] > 0].copy()
    print(f'[V5] Zero-Tage aus Training entfernt: {before_zero - len(df_train):,} '
          f'({(before_zero - len(df_train))/max(before_zero,1):.1%})')

# Val & Test (ohne Grauzone)
df_val = daily_full[
    (daily_full['date'] >= train_end) &
    (daily_full['date'] < val_end) &
    (daily_full['label'] != 'grauzone')
].copy()

df_test = daily_full[
    (daily_full['date'] >= val_end) &
    (daily_full['label'] != 'grauzone')
].copy()

print(f'\n═══ Split (Daily) ═══')
print(f'Train (normal, active): {len(df_train):>8,} | bis {train_end.date()}')
print(f'Val   (ohne Grauzone):  {len(df_val):>8,} | '
      f'Anomalien: {(df_val["label"]=="anomal").sum():,} '
      f'({(df_val["label"]=="anomal").mean():.2%})')
print(f'Test  (ohne Grauzone):  {len(df_test):>8,} | '
      f'Anomalien: {(df_test["label"]=="anomal").sum():,} '
      f'({(df_test["label"]=="anomal").mean():.2%})')

# ── Normalisierung ──
scaler = StandardScaler()
train_scaled = scaler.fit_transform(df_train[FEATURE_COLS].values)
val_scaled   = scaler.transform(df_val[FEATURE_COLS].values)
test_scaled  = scaler.transform(df_test[FEATURE_COLS].values)
print(f'Scaler fitted auf {len(train_scaled):,} Trainings-Samples')

# ── Sequenzen (7 Tage pro Window) ──
def create_station_sequences(df, scaled_data, window_size):
    """Erstellt Sequenzen der Länge window_size pro Station."""
    sequences, labels, meta = [], [], []
    station_ids = df['station_id'].values
    timestamps  = df['date'].values
    label_arr   = df['label'].values

    for station in df['station_id'].unique():
        mask = station_ids == station
        s_data   = scaled_data[mask]
        s_labels = label_arr[mask]
        s_times  = timestamps[mask]

        if len(s_data) < window_size:
            continue

        for i in range(len(s_data) - window_size + 1):
            sequences.append(s_data[i:i + window_size])
            labels.append(s_labels[i + window_size - 1])
            meta.append({
                'station_id': station,
                'timestamp': s_times[i + window_size - 1]
            })

    return np.array(sequences, dtype=np.float32), np.array(labels), meta

print(f'\nSequenzen erstellen (Window={WINDOW_SIZE} Tage)...')
X_train, y_train, meta_train = create_station_sequences(df_train, train_scaled, WINDOW_SIZE)
X_val,   y_val,   meta_val   = create_station_sequences(df_val,   val_scaled,   WINDOW_SIZE)
X_test,  y_test,  meta_test  = create_station_sequences(df_test,  test_scaled,  WINDOW_SIZE)

print(f'X_train: {X_train.shape}  (nur normal, active)')
print(f'X_val:   {X_val.shape}    (Anomalien: {(y_val=="anomal").sum()})')
print(f'X_test:  {X_test.shape}   (Anomalien: {(y_test=="anomal").sum()})')

# Für Vanilla AE und VAE: flatten
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_val_flat   = X_val.reshape(X_val.shape[0], -1)
X_test_flat  = X_test.reshape(X_test.shape[0], -1)
print(f'Flattened dim: {INPUT_DIM_FLAT}')

# ── Sanity Checks ──
print(f'\n═══ SANITY CHECKS ═══')
print(f'y_train: {np.unique(y_train, return_counts=True)}')
print(f'y_val:   {np.unique(y_val, return_counts=True)}')
print(f'y_test:  {np.unique(y_test, return_counts=True)}')

assert 'z_score' not in FEATURE_COLS, "z_score darf NICHT in FEATURE_COLS!"
assert 'demand_residual' not in FEATURE_COLS, "demand_residual darf NICHT in Features!"
assert (y_train == 'normal').all(), "y_train enthält nicht-normale Labels!"
assert (y_val == 'anomal').sum() > 0, "Keine Anomalien in Validation!"
assert (y_test == 'anomal').sum() > 0, "Keine Anomalien in Test!"
print('✅ Alle Checks bestanden.\n')

[V5] Zero-Tage aus Training entfernt: 6,125 (10.1%)

═══ Split (Daily) ═══
Train (normal, active):   54,248 | bis 2025-02-28
Val   (ohne Grauzone):    13,804 | Anomalien: 239 (1.73%)
Test  (ohne Grauzone):    15,071 | Anomalien: 99 (0.66%)
Scaler fitted auf 54,248 Trainings-Samples

Sequenzen erstellen (Window=7 Tage)...
X_train: (53701, 7, 19)  (nur normal, active)
X_val:   (13252, 7, 19)    (Anomalien: 237)
X_test:  (14519, 7, 19)   (Anomalien: 99)
Flattened dim: 133

═══ SANITY CHECKS ═══
y_train: (array(['normal'], dtype='<U6'), array([53701]))
y_val:   (array(['anomal', 'normal'], dtype='<U6'), array([  237, 13015]))
y_test:  (array(['anomal', 'normal'], dtype='<U6'), array([   99, 14420]))
✅ Alle Checks bestanden.



In [11]:
# ══════════════════════════════════════════════════════════════
# 9 — Modellarchitekturen
# ══════════════════════════════════════════════════════════════

class VanillaAE(nn.Module):
    def __init__(self, input_dim, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, latent_dim), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, input_dim)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def get_latent(self, x):
        return self.encoder(x)


class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features, latent_dim=32, n_layers=2, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.latent_dim = latent_dim

        self.encoder = nn.LSTM(
            input_size=n_features, hidden_size=latent_dim,
            num_layers=n_layers, batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
        self.decoder = nn.LSTM(
            input_size=latent_dim, hidden_size=latent_dim,
            num_layers=n_layers, batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
        self.output_layer = nn.Linear(latent_dim, n_features)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        _, (hidden, cell) = self.encoder(x)
        latent = hidden[-1].unsqueeze(1).repeat(1, seq_len, 1)
        decoder_out, _ = self.decoder(latent, (hidden, cell))
        return self.output_layer(decoder_out)

    def get_latent(self, x):
        _, (hidden, _) = self.encoder(x)
        return hidden[-1]


class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128), nn.ReLU()
        )
        self.fc_mu     = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, input_dim)
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

    def get_latent(self, x):
        mu, _ = self.encode(x)
        return mu


# ══════════════════════════════════════════════════════════════
# 10 — Training Framework
# ══════════════════════════════════════════════════════════════

def train_ae(model, X_train, X_val, model_name='AE',
             epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE,
             is_vae=False, vae_beta=1.0):

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=7
    )
    criterion = nn.MSELoss()
    scaler_amp = torch.amp.GradScaler('cuda')

    train_tensor = torch.FloatTensor(X_train).to(device)
    train_loader = DataLoader(
        TensorDataset(train_tensor, train_tensor),
        batch_size=batch_size, shuffle=True, drop_last=True,
        num_workers=0, pin_memory=False
    )

    history = {'train_loss': [], 'val_loss': []}
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    EARLY_STOP_PATIENCE = 15

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for batch_x, _ in train_loader:
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                if is_vae:
                    x_hat, mu, logvar = model(batch_x)
                    recon_loss = criterion(x_hat, batch_x)
                    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
                    loss = recon_loss + vae_beta * kl_loss
                else:
                    x_hat = model(batch_x)
                    loss = criterion(x_hat, batch_x)

            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler_amp.step(optimizer)
            scaler_amp.update()
            epoch_loss += loss.item()

        avg_train_loss = epoch_loss / len(train_loader)

        # Validation (alle 3 Epochs)
        if (epoch + 1) % 3 == 0 or epoch == 0 or epoch == epochs - 1:
            model.eval()
            val_losses = []
            with torch.no_grad(), torch.amp.autocast('cuda'):
                for vi in range(0, len(X_val), 4096):
                    v_chunk = torch.FloatTensor(X_val[vi:vi+4096]).to(device)
                    if is_vae:
                        v_hat, v_mu, v_lv = model(v_chunk)
                        v_recon = criterion(v_hat, v_chunk)
                        v_kl = -0.5 * torch.mean(1 + v_lv - v_mu.pow(2) - v_lv.exp())
                        val_losses.append((v_recon + vae_beta * v_kl).item())
                    else:
                        v_hat = model(v_chunk)
                        val_losses.append(criterion(v_hat, v_chunk).item())
                    del v_chunk, v_hat
            val_loss = np.mean(val_losses)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 3
        else:
            val_loss = history['val_loss'][-1] if history['val_loss'] else avg_train_loss

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_loss)
        scheduler.step(val_loss)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            lr_now = optimizer.param_groups[0]['lr']
            print(f'  [{model_name}] Epoch {epoch+1:>3}/{epochs} | '
                  f'Train: {avg_train_loss:.6f} | Val: {val_loss:.6f} | LR: {lr_now:.1e}')

        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f'  [{model_name}] Early stopping at epoch {epoch+1}')
            break

    if best_state:
        model.load_state_dict(best_state)
        model = model.to(device)
    print(f'  [{model_name}] Best Val Loss: {best_val_loss:.6f}\n')
    return history


# ══════════════════════════════════════════════════════════════
# 11 — Scoring & Evaluation
#
# [FIX #6] Scoring über GANZE Sequenz, nicht nur letzte N Stunden
# BEGRÜNDUNG: "Letzte 3h" war ein Hyperparameter, der auf das eigene
# Labeling-Schema optimiert. Die ganze Sequenz ist konservativer
# und ehrlicher. Der AE-Reconstruction-Error über 24h ist ein
# robusteres Anomalie-Signal.
#
# [FIX #4] Demand-only Scoring: Nur die 6 Demand-Features werden
# für den Score herangezogen. Wetter/Kalender sollen rekonstruiert
# werden (hilft dem AE), aber nicht in den Score einfließen.
# ══════════════════════════════════════════════════════════════

def compute_anomaly_scores(model, X, is_vae=False, chunk_size=2048,
                           demand_indices=DEMAND_FEATURE_INDICES):
    """
    Reconstruction Error NUR auf Demand-Features, über die GANZE Sequenz.
    """
    model.eval()
    model = model.to(device)
    all_scores = []

    for i in range(0, len(X), chunk_size):
        chunk = torch.FloatTensor(X[i:i+chunk_size]).to(device)
        with torch.no_grad(), torch.amp.autocast('cuda'):
            if is_vae:
                x_hat, _, _ = model(chunk)
            else:
                x_hat = model(chunk)

            if chunk.dim() == 3:
                # LSTM: (batch, seq_len, features) → Demand-Features filtern
                diff = (chunk[:, :, demand_indices] - x_hat[:, :, demand_indices]) ** 2
            else:
                # Flat: (batch, seq_len*features) → reshape, filtern
                batch_size = chunk.shape[0]
                c_3d = chunk.reshape(batch_size, WINDOW_SIZE, N_FEATURES)
                h_3d = x_hat.reshape(batch_size, WINDOW_SIZE, N_FEATURES)
                diff = (c_3d[:, :, demand_indices] - h_3d[:, :, demand_indices]) ** 2

            # Mean über ganze Sequenz und alle Demand-Features
            scores = diff.mean(dim=tuple(range(1, diff.dim())))

        all_scores.append(scores.cpu())
        del chunk, x_hat, scores, diff
        torch.cuda.empty_cache()

    return torch.cat(all_scores).numpy()


def find_best_threshold(scores, labels):
    """Schwellenwert der den F1-Score auf Val maximiert."""
    binary = (labels == 'anomal').astype(int)
    if binary.sum() == 0:
        return np.percentile(scores, 95), 0.0
    prec, rec, thresholds = precision_recall_curve(binary, scores)
    f1 = 2 * (prec * rec) / (prec + rec + 1e-8)
    best_idx = np.argmax(f1[:-1])
    return thresholds[best_idx], f1[best_idx]


def evaluate_model(model, X, y, threshold, model_name='AE', is_vae=False):
    """Evaluation mit allen Metriken."""
    scores = compute_anomaly_scores(model, X, is_vae=is_vae)
    binary = (y == 'anomal').astype(int)
    preds  = (scores > threshold).astype(int)

    prec, rec, _ = precision_recall_curve(binary, scores)
    auc_pr = auc(rec, prec)
    f1 = f1_score(binary, preds, zero_division=0)

    try:
        auc_roc = roc_auc_score(binary, scores)
    except ValueError:
        auc_roc = 0.0

    print(f'── {model_name} auf Testdaten ──')
    print(f'  AUC-PR:  {auc_pr:.4f}  (Hauptmetrik)')
    print(f'  F1:      {f1:.4f}')
    print(f'  AUC-ROC: {auc_roc:.4f}')
    print(f'  Threshold: {threshold:.6f}')
    print(classification_report(binary, preds, target_names=['Normal', 'Anomal'],
                                zero_division=0))

    return {
        'model': model_name, 'auc_pr': auc_pr, 'f1': f1, 'auc_roc': auc_roc,
        'threshold': threshold, 'scores': scores, 'preds': preds
    }

In [ ]:
print('='*60)
print('MODELL 1: Vanilla Autoencoder (Baseline)')
print('='*60)
torch.cuda.empty_cache()

vanilla_ae = VanillaAE(input_dim=INPUT_DIM_FLAT, latent_dim=LATENT_DIM)
print(f'Parameter: {sum(p.numel() for p in vanilla_ae.parameters()):,}')

hist_vanilla = train_ae(vanilla_ae, X_train_flat, X_val_flat,
                        model_name='VanillaAE', epochs=EPOCHS, batch_size=BATCH_SIZE)

scores_val_v = compute_anomaly_scores(vanilla_ae, X_val_flat)
thresh_v, f1_v = find_best_threshold(scores_val_v, y_val)
print(f'Val-Threshold: {thresh_v:.6f} (F1={f1_v:.4f})')
results_vanilla = evaluate_model(vanilla_ae, X_test_flat, y_test, thresh_v, 'VanillaAE')

vanilla_ae = vanilla_ae.cpu()
torch.cuda.empty_cache()
print('✅ Vanilla AE fertig\n')






In [13]:
print('='*60)
print('MODELL 2: LSTM-Autoencoder')
print('='*60)
torch.cuda.empty_cache()

lstm_ae = LSTMAutoencoder(n_features=N_FEATURES, latent_dim=LATENT_DIM, n_layers=2)
print(f'Parameter: {sum(p.numel() for p in lstm_ae.parameters()):,}')

hist_lstm = train_ae(lstm_ae, X_train, X_val,
                     model_name='LSTM-AE', epochs=EPOCHS, batch_size=2048)

scores_val_l = compute_anomaly_scores(lstm_ae, X_val)
thresh_l, f1_l = find_best_threshold(scores_val_l, y_val)
print(f'Val-Threshold: {thresh_l:.6f} (F1={f1_l:.4f})')
results_lstm = evaluate_model(lstm_ae, X_test, y_test, thresh_l, 'LSTM-AE')

lstm_ae = lstm_ae.cpu()
torch.cuda.empty_cache()
print('✅ LSTM-AE fertig\n')


MODELL 2: LSTM-Autoencoder
Parameter: 32,755
  [LSTM-AE] Epoch   1/50 | Train: 0.980995 | Val: 1.021872 | LR: 1.0e-03
  [LSTM-AE] Epoch   5/50 | Train: 0.491582 | Val: 0.686119 | LR: 1.0e-03
  [LSTM-AE] Epoch  10/50 | Train: 0.390974 | Val: 0.454416 | LR: 1.0e-03
  [LSTM-AE] Epoch  15/50 | Train: 0.240644 | Val: 0.284083 | LR: 1.0e-03
  [LSTM-AE] Epoch  20/50 | Train: 0.194232 | Val: 0.253975 | LR: 1.0e-03
  [LSTM-AE] Epoch  25/50 | Train: 0.168934 | Val: 0.218798 | LR: 1.0e-03
  [LSTM-AE] Epoch  30/50 | Train: 0.151213 | Val: 0.196435 | LR: 1.0e-03
  [LSTM-AE] Epoch  35/50 | Train: 0.136925 | Val: 0.188149 | LR: 1.0e-03
  [LSTM-AE] Epoch  40/50 | Train: 0.125404 | Val: 0.174043 | LR: 1.0e-03
  [LSTM-AE] Epoch  45/50 | Train: 0.115850 | Val: 0.164591 | LR: 1.0e-03
  [LSTM-AE] Epoch  50/50 | Train: 0.107666 | Val: 0.158436 | LR: 1.0e-03
  [LSTM-AE] Best Val Loss: 0.158436

Val-Threshold: 0.104735 (F1=0.0585)
── LSTM-AE auf Testdaten ──
  AUC-PR:  0.0176  (Hauptmetrik)
  F1:      0.0379


In [12]:
print('='*60)
print('MODELL 3: Variational Autoencoder (VAE)')
print('='*60)
torch.cuda.empty_cache()

vae = VAE(input_dim=INPUT_DIM_FLAT, latent_dim=LATENT_DIM)
print(f'Parameter: {sum(p.numel() for p in vae.parameters()):,}')

hist_vae = train_ae(vae, X_train_flat, X_val_flat,
                    model_name='VAE', epochs=EPOCHS, batch_size=BATCH_SIZE,
                    is_vae=True, vae_beta=0.5)

scores_val_vae = compute_anomaly_scores(vae, X_val_flat, is_vae=True)
thresh_vae, f1_vae = find_best_threshold(scores_val_vae, y_val)
print(f'Val-Threshold: {thresh_vae:.6f} (F1={f1_vae:.4f})')
results_vae = evaluate_model(vae, X_test_flat, y_test, thresh_vae, 'VAE', is_vae=True)

vae = vae.cpu()
torch.cuda.empty_cache()
print('✅ VAE fertig\n')

MODELL 3: Variational Autoencoder (VAE)
Parameter: 146,885
  [VAE] Epoch   1/50 | Train: 1.003717 | Val: 1.090526 | LR: 1.0e-03
  [VAE] Epoch   5/50 | Train: 0.656084 | Val: 0.779980 | LR: 1.0e-03
  [VAE] Epoch  10/50 | Train: 0.518031 | Val: 0.599315 | LR: 1.0e-03
  [VAE] Epoch  15/50 | Train: 0.450265 | Val: 0.515805 | LR: 1.0e-03
  [VAE] Epoch  20/50 | Train: 0.425329 | Val: 0.501476 | LR: 1.0e-03
  [VAE] Epoch  25/50 | Train: 0.411426 | Val: 0.483986 | LR: 1.0e-03
  [VAE] Epoch  30/50 | Train: 0.401654 | Val: 0.471867 | LR: 1.0e-03
  [VAE] Epoch  35/50 | Train: 0.396094 | Val: 0.469633 | LR: 1.0e-03
  [VAE] Epoch  40/50 | Train: 0.391126 | Val: 0.467338 | LR: 1.0e-03
  [VAE] Epoch  45/50 | Train: 0.387080 | Val: 0.464383 | LR: 1.0e-03
  [VAE] Epoch  50/50 | Train: 0.383108 | Val: 0.462513 | LR: 1.0e-03
  [VAE] Best Val Loss: 0.462513

Val-Threshold: 0.346262 (F1=0.0613)
── VAE auf Testdaten ──
  AUC-PR:  0.0174  (Hauptmetrik)
  F1:      0.0390
  AUC-ROC: 0.7175
  Threshold: 0.34626